In [1]:
import pandas as pd

## 1. 제조업 경기실사지수(BSI) 정제

반도체·디스플레이·자동차를 고르고 원본에 있는 모든 기간을 정제합니다. 중간 표의 일부 행은 변환 모양을 확인하는 예시입니다. 아래 기간 요약에는 전체 검증 범위가 나옵니다.

In [2]:
# BSI 원본의 모든 기간 열을 읽고, 빈칸도 숫자 검증까지 보존한다.
bsi_raw = pd.read_csv(
    "산업통상부_제조업 경기실사지수동향_20260630.csv",
    encoding="cp949",
    dtype=str,
    keep_default_na=False,
)
print(f"원본 업종 행 {len(bsi_raw)}개, 기간 열 {len(bsi_raw.columns) - 1}개")

원본 업종 행 22개, 기간 열 56개


In [3]:
# 반도체를 중심으로, 반도체를 부품으로 쓰는 디스플레이·자동차의 BSI도 준비한다.
industries = ["반도체", "디스플레이", "자동차"]
industry_name = bsi_raw["업종분류"].str.strip()
bsi_selected = bsi_raw.loc[industry_name.isin(industries)].copy()
bsi_selected["업종분류"] = industry_name.loc[bsi_selected.index]

if bsi_selected["업종분류"].value_counts().to_dict() != dict.fromkeys(industries, 1):
    raise ValueError("반도체·디스플레이·자동차 원본 행이 각각 한 개 있어야 합니다.")

period_columns = bsi_selected.columns.drop("업종분류")
if period_columns.empty:
    raise ValueError("BSI 기간 열을 찾을 수 없습니다.")

print("정제에 사용하는 업종:", ", ".join(bsi_selected["업종분류"]))
print(f"정제에 사용하는 기간 열 {len(period_columns)}개")
print(f"원본 기간 열의 양 끝: {period_columns[0]} ~ {period_columns[-1]}")

정제에 사용하는 업종: 반도체, 디스플레이, 자동차
정제에 사용하는 기간 열 56개
원본 기간 열의 양 끝: 2019-3분기 현황 ~ 2026년-3분기 전망


In [4]:
# 선택한 업종의 전체 기간 열에 빈칸이 있는지 확인한다.
bsi_values = bsi_selected[period_columns].apply(lambda column: column.str.strip())
blank = bsi_values.isna() | bsi_values.eq("")
print("원본 빈칸 수:", int(blank.sum().sum()))

원본 빈칸 수: 0


In [5]:
# 표를 뒤집어 분기·구분을 행으로, 업종을 열로 옮긴다.
bsi_transposed = bsi_selected.set_index("업종분류").T
bsi_transposed.head()

업종분류,반도체,디스플레이,자동차
2019-3분기 현황,73,78,74
2019-4분기 전망,82,82,90
2019-4분기 현황,85,73,87
2020-1분기 전망,89,75,90
2020-1분기 현황,74,78,58


In [6]:
# 업종 열을 세로로 쌓아 업종·분기·구분당 한 행을 만든다.
bsi_named = bsi_transposed.rename_axis(index="source_period", columns="industry")
bsi_long = bsi_named.stack().reset_index(name="bsi")

if len(bsi_long) != len(industries) * len(period_columns):
    raise ValueError("표를 변환하는 동안 원본 BSI 값이 빠졌습니다.")
bsi_long.head()

,source_period,industry,bsi
0,2019-3분기 현황,반도체,73
1,2019-3분기 현황,디스플레이,78
2,2019-3분기 현황,자동차,74
3,2019-4분기 전망,반도체,82
4,2019-4분기 전망,디스플레이,82


In [7]:
# '2019-3분기 현황' 같은 제목에서 연도, 분기 번호, 현황·전망 구분을 꺼낸다.
parts = bsi_long["source_period"].str.strip().str.extract(
    r"^(?P<year>\d{4})년?-(?P<quarter>[1-4])분기\s+(?P<bsi_type>현황|전망)$"
)

if parts.isna().any(axis=1).any():
    invalid = bsi_long.loc[parts.isna().any(axis=1), "source_period"].unique()
    raise ValueError(f"해석할 수 없는 BSI 기간 열: {invalid.tolist()}")
parts.head()

,year,quarter,bsi_type
0,2019,3,현황
1,2019,3,현황
2,2019,3,현황
3,2019,4,전망
4,2019,4,전망


In [8]:
# 분기 번호를 1·4·7·10월로 바꿔 생산지수와 연결할 분기 시작일을 만든다.
first_month = (parts["quarter"].astype(int) - 1) * 3 + 1

bsi_long["quarter_start"] = pd.to_datetime({
    "year": parts["year"].astype(int),
    "month": first_month,
    "day": 1,
})
bsi_long["bsi_type"] = parts["bsi_type"]

bsi_long[["industry", "quarter_start", "bsi_type", "bsi"]].head()

,industry,quarter_start,bsi_type,bsi
0,반도체,2019-07-01,현황,73
1,디스플레이,2019-07-01,현황,78
2,자동차,2019-07-01,현황,74
3,반도체,2019-10-01,전망,82
4,디스플레이,2019-10-01,전망,82


In [9]:
# BSI를 숫자로 바꾸고 결측치, 범위 오류, 업종·분기·구분이 같은 중복을 확인한다.
key = ["industry", "quarter_start", "bsi_type"]

bsi_clean = bsi_long[key + ["bsi"]].copy()
bsi_clean["bsi"] = pd.to_numeric(bsi_clean["bsi"], errors="coerce")

missing = bsi_clean["bsi"].isna()
out_of_range = ~missing & ~bsi_clean["bsi"].between(0, 200)
duplicates = bsi_clean.duplicated(subset=key, keep=False)

print("전체 행 수:", len(bsi_clean))
print("비어 있거나 숫자로 바꿀 수 없는 BSI:", missing.sum())
print("0~200 범위를 벗어난 BSI:", out_of_range.sum())
print("중복에 해당하는 행:", duplicates.sum())

if missing.any() or out_of_range.any() or duplicates.any():
    raise ValueError("BSI 오류를 확인한 뒤 다시 실행하세요.")

전체 행 수: 168
비어 있거나 숫자로 바꿀 수 없는 BSI: 0
0~200 범위를 벗어난 BSI: 0
중복에 해당하는 행: 0


In [10]:
# 현황과 전망이 원본에서 실제로 제공되는 기간을 확인한다.
bsi_clean = bsi_clean.sort_values(key, ignore_index=True)
bsi_coverage = bsi_clean.groupby("bsi_type").agg(
    start=("quarter_start", "min"),
    end=("quarter_start", "max"),
    quarters=("quarter_start", "nunique"),
    rows=("bsi", "size"),
)
bsi_coverage.rename(columns={"start": "시작", "end": "끝", "quarters": "분기 수", "rows": "행 수"})

,시작,끝,분기 수,행 수
bsi_type,,,,
전망,2019-10-01,2026-07-01,28,84
현황,2019-07-01,2026-04-01,28,84


이 원본에서는 2019년 3분기의 **전망**과 2026년 3분기의 **현황**이 기간 열로 제공되지 않습니다. 제공된 열의 빈칸과 구분합니다. 생산지수와 비교할 때는 SQL에서 월 자료가 3개 있는 분기와 BSI 현황이 함께 있는 분기를 연결합니다.

In [11]:
# 엑셀에서도 한글을 읽기 쉽도록 UTF-8 BOM 형식으로 저장한다.
bsi_clean.to_csv(
    "bsi_quarterly_notebook.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

## 2. 월별 반도체 생산지수 정제

In [12]:
# KOSIS 파일의 첫 줄은 월, 둘째 줄은 지표 이름이라 두 줄 모두 열 제목으로 읽는다.
production_raw = pd.read_csv(
    "시도_산업별_광공업생산지수_2020100__20261119114725.csv",
    encoding="utf-8-sig",
    header=[0, 1],
    dtype=str,
)

production_raw.iloc[:5, :6]

,시도별,산업별,2019.04,2019.05,2019.06,2019.07
,시도별,산업별,생산지수(계절조정),생산지수(계절조정),생산지수(계절조정),생산지수(계절조정)
0,전국,반도체 제조업,76.534,76.308,80.799,82.942


In [13]:
# 계절조정 생산지수 열을 고르고, 전국 반도체 제조업 행만 남긴다.
top = production_raw.columns.get_level_values(0)
bottom = production_raw.columns.get_level_values(1)

keep_columns = top.isin(["시도별", "산업별"]) | (bottom == "생산지수(계절조정)")
production = production_raw.loc[:, keep_columns].copy()
production.columns = production.columns.get_level_values(0)

is_national = production["시도별"].str.strip() == "전국"
is_semiconductor = production["산업별"].str.strip() == "반도체 제조업"

production_selected = production.loc[is_national & is_semiconductor].copy()
production_selected.iloc[:, :6]

,시도별,산업별,2019.04,2019.05,2019.06,2019.07
0,전국,반도체 제조업,76.534,76.308,80.799,82.942


In [14]:
# 가로로 놓인 월별 값을 뒤집어 한 달에 한 행씩 배치한다.
monthly_values = production_selected.drop(columns=["시도별", "산업별"])

production_long = monthly_values.T
production_long.columns = ["production_index"]
production_long = production_long.rename_axis("source_month").reset_index()

production_long.head()

,source_month,production_index
0,2019.04,76.534
1,2019.05,76.308
2,2019.06,80.799
3,2019.07,82.942
4,2019.08,83.754


In [15]:
# '2026.09 p)'에서 월 시작일과 잠정치 표시를 각각 저장한다.
source_month = production_long["source_month"].str.strip()

production_long["is_provisional"] = (
    source_month.str.endswith("p)", na=False).astype(int)
)

month_text = source_month.str.replace(r"\s+p\)$", "", regex=True)
production_long["month_start"] = pd.to_datetime(month_text, format="%Y.%m")

production_long[["source_month", "month_start", "is_provisional"]].tail()

,source_month,month_start,is_provisional
85,2026.05,2026-05-01,0
86,2026.06,2026-06-01,0
87,2026.07,2026-07-01,0
88,2026.08 p),2026-08-01,1
89,2026.09 p),2026-09-01,1


In [16]:
# 최근 행을 보면서 월 시작일과 잠정치가 제대로 분리됐는지 확인한다.
production_long[["month_start", "production_index", "is_provisional"]].tail()

,month_start,production_index,is_provisional
85,2026-05-01,182.4,0
86,2026-06-01,189.8,0
87,2026-07-01,191.4,0
88,2026-08-01,192.8,1
89,2026-09-01,229.9,1


In [17]:
# 생산지수를 숫자로 바꾸고, BSI와 연결할 수 있도록 업종 이름을 '반도체'로 맞춘다.
production_clean = production_long[
    ["month_start", "production_index", "is_provisional"]
].copy()

production_clean["production_index"] = pd.to_numeric(
    production_clean["production_index"], errors="coerce"
)

production_clean["region"] = "전국"
production_clean["industry"] = "반도체"

# 지역·업종·월이 같은 행을 중복으로 보고, 결측치와 잠정치 수도 확인한다.
key = ["region", "industry", "month_start"]
production_clean = production_clean[key + ["production_index", "is_provisional"]]

print("전체 행 수:", len(production_clean))
print("날짜 결측:", production_clean["month_start"].isna().sum())
print("생산지수 결측:", production_clean["production_index"].isna().sum())
print("중복에 해당하는 행:", production_clean.duplicated(subset=key, keep=False).sum())
print("잠정치 행 수:", production_clean["is_provisional"].sum())

production_clean.head()

전체 행 수: 90
날짜 결측: 0
생산지수 결측: 0
중복에 해당하는 행: 0
잠정치 행 수: 2


,region,industry,month_start,production_index,is_provisional
0,전국,반도체,2019-04-01,76.534,0
1,전국,반도체,2019-05-01,76.308,0
2,전국,반도체,2019-06-01,80.799,0
3,전국,반도체,2019-07-01,82.942,0
4,전국,반도체,2019-08-01,83.754,0


In [18]:
# 월별 정제본을 저장하고, 분기 평균은 SQL 분석 단계에서 계산한다.
production_clean.to_csv(
    "production_monthly_notebook.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)